# Feed loader – CSV → Delta (append)
Python notebook (not PySpark). Loads one CSV per feed into its own Delta table in the Lakehouse, appending each run.

**Per run:** find new CSV(s) in each feed's landing folder → read as text and cast to explicit types (Int32/Int64, Decimal(18,2), Date, UTC Datetime) → add audit columns → skip if this file was already loaded → append to Delta → move CSV to `archive/`.

**Feeds are discovered**: every child folder of `LANDING_ROOT` is a feed, loaded into its own table, with column types from a `_schema.json` in the folder. Or pass one `FOLDER` (and optionally `TABLE`) to load a single location.

Within a feed, files are loaded **one at a time**, so memory depends on the largest single file, not on how many files there are. Set `PARALLEL_FEEDS` above 1 to load several feeds at once in this same session (one thread per feed); `FEED_FILTER` limits a run to named feeds. A failure in one feed stops that feed only; the others carry on, and the run fails at the end so a pipeline sees it. Files at or above `STREAM_THRESHOLD_MB` are **streamed** (`scan_csv` → `sink_delta`) and never held in memory whole; smaller files are loaded in memory, which also gives an exact row count before the write. On Polars versions without `sink_delta`, large files fall back to an in-memory load (`stream-fallback` in the summary).

Writes use row groups of up to `ROW_GROUP_ROWS` rows (8 million by default), which suits Power BI Direct Lake better than the ~1 million default. delta-rs can't apply V-Order; that needs Spark.

## Optional: pin library versions
Fabric's preinstalled Polars may be older than 2.0. Older versions still load everything, but files at or above `STREAM_THRESHOLD_MB` are then read whole into memory instead of streamed (the version check below says which you have). To get true streaming, uncomment the line below. It must run before anything imports Polars, and it adds start-up time to every run.

In [ ]:
# %pip install -q "polars==2.0.0" "deltalake==1.6.6"


In [ ]:
# Parameters (mark this cell as a parameter cell so a pipeline can override)
WORKSPACE = "MyWorkspace"
LAKEHOUSE = "MyLakehouse"
SCHEMA    = "dbo"      # set to None if the Lakehouse is not schema-enabled
DRY_RUN   = False
STREAM_THRESHOLD_MB = 500   # files this size or larger are streamed to Delta
LANDING_ROOT   = "landing"  # parent folder under Files; each child folder is a feed
ARCHIVE_ROOT   = "archive"  # loaded CSVs move to ARCHIVE_ROOT/<feed>
FOLDER         = ""         # "" = discover every child of LANDING_ROOT; or one folder, e.g. "landing/orders" or "incoming/acme/orders"
TABLE          = ""         # with FOLDER: target table name (overrides the schema file and the default)
TABLE_SUFFIX   = "_raw"     # default table name = folder name + suffix
MISSING_SCHEMA = "skip"     # folder without _schema.json: "skip" (flag it), "fail", or "text" (load every column as text)
FEED_FILTER    = ""         # with discovery: only these feeds, e.g. "customers" / "orders,products"
PARALLEL_FEEDS = 1          # feeds loaded at once in this session (1 = one after another)
ROW_GROUP_ROWS = 8_000_000  # max rows per Parquet row group; larger groups suit Direct Lake. None = library default (~1M)
LOG_TABLE      = "load_log" # Delta table that records every run (one row per file); None = don't log

In [ ]:
import gc
import polars as pl
import deltalake
import notebookutils
from datetime import datetime, timezone
from deltalake import DeltaTable

# --- Version check: which features this Polars has --------------------------
HAS_SINK_DELTA = hasattr(pl.LazyFrame, "sink_delta")   # true streaming to Delta

def collect_streaming(lf):
    """Run a lazy query with the streaming engine on any Polars version."""
    try:
        return lf.collect(engine="streaming")     # Polars 1.2x and later
    except (TypeError, ValueError):
        return lf.collect(streaming=True)         # older Polars

print(f"polars {pl.__version__}, deltalake {deltalake.__version__}")
if not HAS_SINK_DELTA:
    print("NOTE: this Polars has no sink_delta, so large files are read whole into memory "
          "instead of streamed. Pin a newer version in the cell above for true streaming.")

# --- Paths and auth -----------------------------------------------------------
def _enc(name):
    return name.replace(" ", "%20")   # spaces in workspace/lakehouse names must be encoded

BASE = f"abfss://{_enc(WORKSPACE)}@onelake.dfs.fabric.microsoft.com/{_enc(LAKEHOUSE)}.Lakehouse"

def storage_options():
    # fetched per call so long runs don't hit token expiry
    return {
        "bearer_token": notebookutils.credentials.getToken("storage"),
        "use_fabric_endpoint": "true",
    }

def table_path(name):
    return f"{BASE}/Tables/{SCHEMA}/{name}" if SCHEMA else f"{BASE}/Tables/{name}"

def files_path(rel):
    return f"{BASE}/Files/{rel}"

## Feed discovery and schema files
Each child folder of `LANDING_ROOT` is a feed. Its table is named after the folder (`orders` → `orders_raw`) and loaded CSVs move to `ARCHIVE_ROOT/<folder>`. Column types come from a `_schema.json` file in the folder:

```json
{
  "table": "orders_raw",
  "columns": {"order_id": "Int64", "customer_id": "Int32", "order_date": "Date", "amount": "Decimal(18,2)"},
  "formats": {"order_date": "%Y-%m-%d"},
  "csv_options": {"separator": ",", "has_header": true}
}
```

`columns` lists every CSV column in file order. Types: `String`, `Int8`–`Int64`, `UInt8`–`UInt64`, `Float32`, `Float64`, `Boolean`, `Date`, `Datetime` (stored as UTC, microseconds) and `Decimal(p,s)`. `Date` and `Datetime` columns need a format in `formats`. `table` and `csv_options` are optional.

In [ ]:
import json, re

TYPE_MAP = {
    "string": pl.Utf8, "utf8": pl.Utf8, "boolean": pl.Boolean, "bool": pl.Boolean,
    "int8": pl.Int8, "int16": pl.Int16, "int32": pl.Int32, "int64": pl.Int64,
    "uint8": pl.UInt8, "uint16": pl.UInt16, "uint32": pl.UInt32, "uint64": pl.UInt64,
    "float32": pl.Float32, "float64": pl.Float64, "date": pl.Date,
    "datetime": pl.Datetime("us", "UTC"),
}

def parse_type(name, where):
    key = name.strip().lower().replace(" ", "")
    m = re.fullmatch(r"decimal\((\d+),(\d+)\)", key)
    if m:
        return pl.Decimal(int(m.group(1)), int(m.group(2)))
    if key in TYPE_MAP:
        return TYPE_MAP[key]
    raise ValueError(f"{where}: unknown type '{name}'")

def safe_name(folder):
    """Folder name -> table/feed name: lower case, letters, digits and underscores."""
    return re.sub(r"[^a-z0-9_]+", "_", folder.strip().lower()).strip("_")

def read_schema_file(rel_folder):
    """Parsed _schema.json for a folder, or None if it has none."""
    path = files_path(f"{rel_folder}/_schema.json")
    try:
        text = notebookutils.fs.head(path, 1_000_000)
    except Exception:
        return None
    where = f"{rel_folder}/_schema.json"
    spec = json.loads(text)
    if not spec.get("columns"):
        raise ValueError(f"{where}: 'columns' is missing or empty")
    columns = {c: parse_type(t, f"{where} column '{c}'") for c, t in spec["columns"].items()}
    formats = spec.get("formats", {})
    for c, t in columns.items():
        if (t == pl.Date or isinstance(t, pl.Datetime)) and c not in formats:
            raise ValueError(f"{where}: column '{c}' is {t} but has no entry in 'formats'")
    return {"table": spec.get("table"), "columns": columns, "formats": formats,
            "csv_options": spec.get("csv_options", {"separator": ",", "has_header": True})}

def build_feed(rel_folder, table_override=""):
    name = safe_name(rel_folder.rstrip("/").rsplit("/", 1)[-1])
    spec = read_schema_file(rel_folder)
    if spec is None:
        if MISSING_SCHEMA == "text":
            spec = {"table": None, "columns": None, "formats": {},
                    "csv_options": {"separator": ",", "has_header": True}}
        else:
            spec = None
    # archive mirrors the landing path: landing/orders -> archive/orders, incoming/acme/orders -> archive/incoming/acme/orders
    rel_under_root = rel_folder[len(LANDING_ROOT) + 1:] if rel_folder.startswith(LANDING_ROOT + "/") else rel_folder
    feed = {"landing": rel_folder, "archive": f"{ARCHIVE_ROOT}/{rel_under_root}",
            "table": table_override or (spec or {}).get("table") or f"{name}{TABLE_SUFFIX}",
            "schema_source": "_schema.json" if spec and spec["columns"] else
                             ("all text" if spec else "missing")}
    if spec:
        feed.update(columns=spec["columns"], formats=spec["formats"], csv_options=spec["csv_options"])
    return name, feed

def subfolders(rel_folder):
    try:
        return sorted(f.name for f in notebookutils.fs.ls(files_path(rel_folder)) if f.isDir)
    except Exception:
        return []

# --- Work out the feeds for this run ---------------------------------------
if FOLDER:
    folder = FOLDER.strip("/")
    FEEDS = dict([build_feed(folder, TABLE)])
else:
    if TABLE:
        raise ValueError("TABLE can only be used together with FOLDER")
    FEEDS = dict(build_feed(f"{LANDING_ROOT}/{d}") for d in subfolders(LANDING_ROOT))
    if FEED_FILTER:
        wanted = {safe_name(f) for f in FEED_FILTER.split(",") if f.strip()}
        unknown = wanted - FEEDS.keys()
        if unknown:
            raise ValueError(f"FEED_FILTER names feeds with no folder under {LANDING_ROOT}: {sorted(unknown)}")
        FEEDS = {k: v for k, v in FEEDS.items() if k in wanted}

# two folders must never load into the same table
tables = {}
for name, f in FEEDS.items():
    if f["table"] in tables:
        raise ValueError(f"Folders '{tables[f['table']]}' and '{f['landing']}' both map to table {f['table']}")
    tables[f["table"]] = f["landing"]

# folders without a schema file
missing = {k: v for k, v in FEEDS.items() if v["schema_source"] == "missing"}
if missing and MISSING_SCHEMA == "fail":
    raise ValueError("No _schema.json in: " + ", ".join(v["landing"] for v in missing.values()))
SKIPPED = [{"feed": k, "folder": v["landing"], "target_table": v["table"], "file": "", "size_mb": None,
            "mode": "", "status": "no schema", "rows": 0, "started_utc": None, "finished_utc": None,
            "seconds": 0.0, "error": f"no _schema.json in {v['landing']}; files left in place"}
           for k, v in missing.items()]
FEEDS = {k: v for k, v in FEEDS.items() if k not in missing}

for k, v in FEEDS.items():
    print(f"{k:<20} {v['landing']:<35} -> {v['table']:<25} ({v['schema_source']})")
for r in SKIPPED:
    print(f"{r['feed']:<20} SKIPPED: {r['error']}")
if not FEEDS:
    print("No feeds to load.")

## Helpers

In [ ]:
def list_csvs(rel_folder):
    try:
        items = notebookutils.fs.ls(files_path(rel_folder))
    except Exception:
        return []
    # (path, size in MB), sorted by name
    return sorted(
        (f.path, f.size / 1024**2) for f in items if f.name.lower().endswith(".csv")
    )

def rows_for_file(target, file_name):
    """Rows in the table from this source file (0 if the table doesn't exist yet).
    Fast even on large tables: every row from one file has the same _source_file,
    so delta-rs skips other files' Parquet using their min/max stats."""
    if not DeltaTable.is_deltatable(target, storage_options=storage_options()):
        return 0
    return (
        pl.scan_delta(target, storage_options=storage_options())
          .filter(pl.col("_source_file") == file_name)
          .select(pl.len())
          .collect()
          .item()
    )

def already_loaded(target, file_name):
    """Idempotency guard: has this source file already been appended?"""
    return rows_for_file(target, file_name) > 0

def scan_feed_csv(path, cfg):
    """Lazy, typed read of one CSV. Everything is read as text, then cast
    strictly, so a bad value fails the load instead of changing a column type."""
    cols = cfg["columns"]
    if cols is None:      # MISSING_SCHEMA = "text": keep the file's own header, every column as text
        return pl.scan_csv(path, infer_schema=False, storage_options=storage_options(),
                           null_values=["", "NULL", "null"], **cfg["csv_options"])
    lf = pl.scan_csv(
        path,
        schema={c: pl.Utf8 for c in cols},
        storage_options=storage_options(),
        null_values=["", "NULL", "null"],
        **cfg["csv_options"],
    )
    exprs = []
    for c, dtype in cols.items():
        e = pl.col(c).str.strip_chars()
        if dtype == pl.Date:
            e = e.str.to_date(cfg["formats"][c], strict=True)
        elif isinstance(dtype, pl.Datetime):
            e = e.str.to_datetime(cfg["formats"][c], time_unit="us", strict=True)
            if dtype.time_zone:
                e = e.dt.replace_time_zone(dtype.time_zone)
        elif dtype == pl.Boolean:
            e = e.str.to_lowercase().replace_strict(
                {"true": True, "false": False, "1": True, "0": False, "yes": True, "no": False},
                return_dtype=pl.Boolean)
        elif dtype != pl.Utf8:
            e = e.cast(dtype, strict=True)
        exprs.append(e.alias(c))
    return lf.select(exprs)

def add_audit(lf, file_name, load_ts):
    return lf.with_columns(
        pl.lit(file_name).alias("_source_file"),
        pl.lit(load_ts).cast(pl.Datetime("us", "UTC")).alias("_load_ts"),
    )

## Load

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

load_ts = datetime.now(timezone.utc)
WRITE_OPTS = {"schema_mode": "merge"}   # allow new source columns
if ROW_GROUP_ROWS:
    from deltalake import WriterProperties
    WRITE_OPTS["writer_properties"] = WriterProperties(max_row_group_size=ROW_GROUP_ROWS)

def utcnow():
    return datetime.now(timezone.utc)

def result_row(feed, cfg, file="", size_mb=None, mode="", status="", rows=0,
               started=None, finished=None, error=None):
    return {"feed": feed, "folder": cfg["landing"], "target_table": cfg["table"], "file": file,
            "size_mb": None if size_mb is None else round(size_mb, 1), "mode": mode,
            "status": status, "rows": rows, "started_utc": started, "finished_utc": finished,
            "seconds": round((finished - started).total_seconds(), 1) if started and finished else 0.0,
            "error": error}

def load_feed(feed, cfg):
    """Load every new CSV for one feed, one file at a time.
    Returns one result row per file. A failure stops this feed only:
    the failed file stays in landing and later files wait for the next run."""
    target = table_path(cfg["table"])
    files = list_csvs(cfg["landing"])
    if not files:
        print(f"[{feed}] no new CSVs in {cfg['landing']}")
        now = utcnow()
        return [result_row(feed, cfg, status="no new files", started=now, finished=now)]
    out = []
    for src, size_mb in files:
        file_name = src.rsplit("/", 1)[-1]
        stream = size_mb >= STREAM_THRESHOLD_MB
        mode = ("stream" if HAS_SINK_DELTA else "stream-fallback") if stream else "memory"
        started = utcnow()
        try:
            status, rows = load_file(feed, cfg, target, src, file_name, size_mb, stream, mode)
        except Exception as e:
            msg = str(e).splitlines()[0][:1000]
            print(f"[{feed}] {file_name} FAILED: {msg}")
            out.append(result_row(feed, cfg, file_name, size_mb, mode, "failed", 0, started, utcnow(), msg))
            break
        out.append(result_row(feed, cfg, file_name, size_mb, mode, status, rows, started, utcnow()))
    return out

def load_file(feed, cfg, target, src, file_name, size_mb, stream, mode):
        if already_loaded(target, file_name):
            print(f"[{feed}] {file_name} already loaded – archiving only")
            status, rows = "skipped", 0
        else:
            lf = add_audit(scan_feed_csv(src, cfg), file_name, load_ts)

            if DRY_RUN:
                # null_count forces every cast to run, so bad values still fail here
                check = lf.select(
                    [pl.len().alias("_rows"), pl.all().null_count()]
                )
                rows = collect_streaming(check)["_rows"][0]
                status = "dry run"
            elif stream and HAS_SINK_DELTA:
                # never holds the whole file in memory; one Delta commit at the end
                lf.sink_delta(target, mode="append",
                              storage_options=storage_options(),
                              delta_write_options=WRITE_OPTS)
                rows = rows_for_file(target, file_name)
                status = "loaded"
            else:
                # in memory; large files on older Polars also come here (streaming engine for the read)
                df = collect_streaming(lf) if stream else lf.collect()
                rows = df.height
                df.write_delta(target, mode="append",
                               storage_options=storage_options(),
                               delta_write_options=WRITE_OPTS)
                del df                      # free this file before reading the next
                status = "loaded"

            del lf
            gc.collect()
            verb = "checked" if DRY_RUN else "appended"
            print(f"[{feed}] {file_name} ({size_mb:,.0f} MB, {mode}): {verb} {rows:,} rows -> {cfg['table']}")

        if not DRY_RUN:
            notebookutils.fs.mv(src, files_path(f"{cfg['archive']}/{file_name}"), True, True)  # create_path, overwrite
        return status, rows

# Run the feeds: one after another, or several at once on threads.
# Polars and delta-rs release Python's GIL while they work, so threads run in parallel.
run_started_utc = utcnow()
workers = max(1, min(PARALLEL_FEEDS, len(FEEDS) or 1))
if workers == 1:
    per_feed = [load_feed(feed, cfg) for feed, cfg in FEEDS.items()]
else:
    with ThreadPoolExecutor(max_workers=workers, thread_name_prefix="feed") as pool:
        per_feed = list(pool.map(lambda kv: load_feed(*kv), FEEDS.items()))
results = [r for feed_rows in per_feed for r in feed_rows] + SKIPPED
run_finished_utc = utcnow()
print(f"Run finished in {(run_finished_utc - run_started_utc).total_seconds():,.0f}s "
      f"with {workers} feed(s) at a time")

TS = pl.Datetime("us", "UTC")
summary = pl.DataFrame(results, schema={
    "feed": pl.Utf8, "folder": pl.Utf8, "target_table": pl.Utf8, "file": pl.Utf8,
    "size_mb": pl.Float64, "mode": pl.Utf8, "status": pl.Utf8, "rows": pl.Int64,
    "started_utc": TS, "finished_utc": TS, "seconds": pl.Float64, "error": pl.Utf8})
summary.select("feed", "file", "mode", "status", "rows", "started_utc", "seconds", "error")

## Load log
Appends this run's results to the `LOG_TABLE` Delta table: one row per file (plus one per feed with no new files or no schema), with the run's ID, start and end times, where it ran from, and the library versions. Dry runs aren't logged. The log is written even when a feed failed, so failures are recorded too.

In [ ]:
import json, uuid

def run_context():
    """Notebook and pipeline details from Fabric, where available."""
    try:
        ctx = dict(notebookutils.runtime.context)
    except Exception:
        ctx = {}
    pick = lambda *keys: next((str(ctx[k]) for k in keys if ctx.get(k) not in (None, "")), None)
    return {
        "notebook_name": pick("currentNotebookName"),
        "triggered_by_pipeline": str(ctx.get("isForPipeline", "")).lower() == "true",
        "pipeline_run_id": pick("pipelineJobId", "pipelineRunId"),
        "user_name": pick("userName"),
        "context_json": json.dumps(ctx, default=str) if ctx else None,
    }

RUN_ID = str(uuid.uuid4())
LOG_ERROR = None

if LOG_TABLE and not DRY_RUN:
    ctx = run_context()
    log_df = summary.with_columns(
        pl.lit(RUN_ID).alias("run_id"),
        pl.lit(run_started_utc).cast(TS).alias("run_started_utc"),
        pl.lit(run_finished_utc).cast(TS).alias("run_finished_utc"),
        pl.lit(WORKSPACE).alias("workspace"),
        pl.lit(LAKEHOUSE).alias("lakehouse"),
        pl.lit(ctx["notebook_name"], dtype=pl.Utf8).alias("notebook_name"),
        pl.lit(ctx["triggered_by_pipeline"]).alias("triggered_by_pipeline"),
        pl.lit(ctx["pipeline_run_id"], dtype=pl.Utf8).alias("pipeline_run_id"),
        pl.lit(ctx["user_name"], dtype=pl.Utf8).alias("user_name"),
        pl.lit(FOLDER or LANDING_ROOT).alias("source_root"),
        pl.lit(workers).cast(pl.Int32).alias("parallel_feeds"),
        pl.lit(pl.__version__).alias("polars_version"),
        pl.lit(deltalake.__version__).alias("deltalake_version"),
        pl.lit(ctx["context_json"], dtype=pl.Utf8).alias("context_json"),
    ).select(
        "run_id", "run_started_utc", "run_finished_utc", "workspace", "lakehouse",
        "feed", "folder", "target_table", "file", "size_mb", "mode", "status", "rows",
        "started_utc", "finished_utc", "seconds", "error",
        "notebook_name", "triggered_by_pipeline", "pipeline_run_id", "user_name",
        "source_root", "parallel_feeds", "polars_version", "deltalake_version", "context_json",
    )
    try:
        log_df.write_delta(table_path(LOG_TABLE), mode="append",
                           storage_options=storage_options(),
                           delta_write_options={"schema_mode": "merge"})
        print(f"Logged {log_df.height} row(s) to {LOG_TABLE} (run_id {RUN_ID})")
    except Exception as e:
        LOG_ERROR = str(e).splitlines()[0][:500]
        print(f"WARNING: could not write the load log: {LOG_ERROR}")
elif DRY_RUN:
    print("Dry run: load log not written")

## Maintenance (optional)
Appends add files every run. Compact periodically – e.g. run this cell weekly or after a large batch, or move it to its own scheduled notebook.

In [ ]:
RUN_MAINTENANCE = False

if RUN_MAINTENANCE:
    for cfg in FEEDS.values():
        t = table_path(cfg["table"])
        if DeltaTable.is_deltatable(t, storage_options=storage_options()):
            dt = DeltaTable(t, storage_options=storage_options())
            print(cfg["table"], dt.optimize.compact())
            dt.vacuum(retention_hours=168, dry_run=False, enforce_retention_duration=True)

In [ ]:
# Fail the run if any feed (or the log write) failed, so a pipeline marks it failed;
# otherwise return the summary with the run ID
failed = summary.filter(pl.col("status") == "failed")
problems = [f"{r['feed']}/{r['file']}: {r['error']}" for r in failed.iter_rows(named=True)]
if LOG_ERROR:
    problems.append(f"load log: {LOG_ERROR}")
if problems:
    raise RuntimeError(f"{len(problems)} problem(s): " + "; ".join(problems))
notebookutils.notebook.exit(json.dumps({
    "run_id": RUN_ID,
    "files": json.loads(summary.drop("started_utc", "finished_utc").write_json()),
}))